In [26]:
import math
import os
import gurobipy as gp
import contextlib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import random
import time
import itertools

from typing import Optional, Dict, List
from openpyxl import load_workbook

Node = str
Veh = str
Route = str

class E2EVRP:
    def __init__(self):
        # Name
        self.instance_name: Optional[str] = None

        # Nodes
        self.N: List[Node] = []
        self.Nd: Optional[Node] = None 
        self.NS: List[Node] = []
        self.NC: List[Node] = []
        self.NR: List[Node] = []

        # Arc parameters
        self.d: Dict[tuple[Node, Node], float] = {}
        self.dr: Dict[tuple[Node, Node, Node], float] = {}
        self.t: Dict[tuple[Node, Node], float] = {}

        # Customer parameters
        self.q: Dict[Node, float] = {}
        self.S: Dict[Node, float] = {}
        self.e: Dict[Node, float] = {}
        self.l: Dict[Node, float] = {}
        self.p: Dict[Node, float] = {}

        # Vehicles
        self.V: List[Veh] = []
        self.T: List[Veh] = []
        self.F: List[Veh] = []

        # Vehicle parameters
        self.Q: Dict[Veh, float] = {}
        self.L: Dict[Veh, float] = {}
        self.h: Dict[Veh, float] = {}
        self.c: Dict[Veh, float] = {}
        self.r: Dict[Veh, float] = {}

        # Vehicle types
        self.VT: List[str] = []
        self.TT: List[str] = []
        self.FT: List[str] = []
        self.Vq: Dict[str, int] = {}
        self.Qv: Dict[str, float] = {}
        self.Lv: Dict[str, float] = {}
        self.hv: Dict[str, float] = {}
        self.cv: Dict[str, float] = {}
        self.rv: Dict[str, float] = {}

        # Constant parameters
        self.eta: Optional[float] = None
        self.T0: Optional[float] = None

        # Decision variables
        self.w: Dict[tuple[Node, Node], float] = dict()
        self.x0: Dict[tuple[Node, Node, Veh], float] = dict()
        self.y0: Dict[Veh, float] = dict()
        self.x: Dict[tuple[Node, Node, Node, Node, Veh], float] = dict()
        self.y: Dict[tuple[Veh, Node], float] = dict()

        # Routes
        self.R: Dict[Node, List[Route]] = dict()
        self.Rd: Dict[tuple[Route, Node], List[Node]] = dict()
        self.C: Dict[tuple[Route, Node], float] = dict()
        self.b: Dict[tuple[Route, str, Node], float] = dict()
        self.A: Dict[tuple[Node, Route, Node], float] = dict()

        # Matheuristica
        self.p1: Optional[float] = 0.25
        self.p2: Optional[float] = 0.25
        self.max1: Optional[int] = 100
        self.max2: Optional[int] = 25

    def read_instance(self, folder, name):
        section = None
        coord = {}

        instance_path = f"./e-2e-vrp instances/{folder}/{name}.txt"
        if not os.path.exists(instance_path):
            raise FileNotFoundError(f"{instance_path} was not found.")

        self.instance_name = name

        with open(instance_path, "r", encoding="utf-8") as f:
            for raw in f:
                line = raw.strip()

                if not line or line.startswith("!--"):
                    continue

                if line.startswith("!Stores"):
                    section = "stores"; continue
                if line.startswith("!Trucks"):
                    section = "trucks"; continue
                if line.startswith("!CityFreighters"):
                    section = "cityf"; continue
                if line.startswith("!Customers"):
                    section = "cust"; continue
                if line.startswith("!Recharge stations"):
                    section = "recharge"; continue

                if line.startswith("g "):
                    parts = line.strip().split()
                    self.eta = float(parts[-1])
                    continue
                if line.startswith("T "):
                    parts = line.strip().split()
                    self.T0 = float(parts[-1])
                    continue

                if section == "stores":
                    items = line.split()
                    for idx, item in enumerate(items):
                        x, y = item.split(",")
                        if idx == 0: # Deposit
                            self.Nd = "D0"
                            coord[self.Nd] = (float(x), float(y))
                            self.N.append(self.Nd)
                        else: #Satellites
                            s_id = f"S{idx-1}"
                            coord[s_id] = (float(x), float(y))
                            self.N.append(s_id)
                            self.NS.append(s_id)
                elif section == "trucks":
                    items = line.split()
                    v_count = 0
                    for idx, item in enumerate(items):
                        tn, tq, tc, th = item.split(",")
                        tn, tq, tc, th = int(tn), int(tq), int(tc), int(th)
                        ttype = f"TT{idx}"
                        self.Vq[ttype] = tn
                        self.Qv[ttype] = tq
                        self.hv[ttype] = th
                        self.cv[ttype] = tc
                        self.VT.append(ttype)
                        self.TT.append(ttype)
                        for i in range(tn):
                            v_id = f"T{i+v_count}"
                            self.Q[v_id] = tq
                            self.h[v_id] = th
                            self.c[v_id] = tc
                            self.V.append(v_id)
                            self.T.append(v_id)
                        v_count += tn
                elif section == "cityf":
                    items = line.split()
                    v_count = 0
                    for idx, item in enumerate(items):
                        _, fn, fq, fc, fh, fb, fr = item.split(",")
                        fn, fq, fc, fh, fb, fr = int(fn), int(fq), int(fc), int(fh), int(fb), int(fr)
                        ftype = f"FT{idx}"
                        self.Vq[ftype] = fn
                        self.Qv[ftype] = fq
                        self.hv[ftype] = fh
                        self.cv[ftype] = fc
                        self.Lv[ftype] = fb
                        self.rv[ftype] = fr
                        self.VT.append(ftype)
                        self.FT.append(ftype)
                        for i in range(fn):
                            v_id = f"F{i+v_count}"
                            self.Q[v_id] = fq
                            self.h[v_id] = fh
                            self.c[v_id] = fc
                            self.L[v_id] = fb
                            self.r[v_id] = fr
                            self.V.append(v_id)
                            self.F.append(v_id)
                        v_count += fn
                elif section == "cust":
                    items = line.split()
                    for idx, item in enumerate(items):
                        x, y, d, rt, dd, st, pc = item.split(",")
                        c_id = f"C{idx}"
                        coord[c_id] = (float(x), float(y))
                        self.q[c_id] = float(d)
                        self.e[c_id] = float(rt)
                        self.l[c_id] = float(dd)
                        self.S[c_id] = float(st)
                        self.p[c_id] = float(pc)
                        self.N.append(c_id)
                        self.NC.append(c_id)
                elif section == "recharge":
                    items = line.split()
                    for idx, item in enumerate(items):
                        x, y = item.split(",")
                        r_id = f"R{idx}"
                        coord[r_id] = (float(x), float(y))
                        self.N.append(r_id)
                        self.NR.append(r_id)
                else:
                    continue
            
        for i in self.N:
            xi, yi = coord[i]
            for j in self.N:
                xj, yj = coord[j]
                dij = round(math.sqrt((xi-xj)**2 + (yi-yj)**2))
                self.d[i,j] = dij
                self.t[i,j] = dij

        for i in self.NS+self.NC:
            xi, yi = coord[i]
            for j in self.NS+self.NC:
                self.dr[i,j,'0'] = self.d[i,j]
                for r in self.NR:
                    self.dr[i,j,r] = self.d[i,r] + self.d[r,j]

    def milp_model(self, env_params, time_limit, output=0):
        M = 3*max(self.l.values())
        Mq = 3*max(self.Q.values())
        Ml = 3*max(self.L.values())
        Nd_NS = [self.Nd]+self.NS
        Rset = ['0']+self.NR

        with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            env = gp.Env(params=env_params)

        #Model
        model = gp.Model(self.instance_name, env=env)
        model.Params.OutputFlag = output
        model.Params.TimeLimit = time_limit

        #Variables
        w = {(i,s): model.addVar(vtype = gp.GRB.BINARY, name = f"w^{s}({i})") for s in self.NS for i in self.NC}
        a = {s: model.addVar(vtype = gp.GRB.BINARY, name = f"a({s})") for s in self.NS}
        x0 = {(i,j,v): model.addVar(vtype = gp.GRB.BINARY, name = f"x^0({i},{j},{v})") for i in Nd_NS for j in Nd_NS for v in self.T if i!=j}
        y0 = {v: model.addVar(vtype = gp.GRB.BINARY, name = f"y^0({v})") for v in self.T}
        U0 = {(i,v): model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"U({i},{v})") for i in Nd_NS for v in self.T}
        x = {(i,j,r,v,s): model.addVar(vtype = gp.GRB.BINARY, name = f"x^{s}({i},{j},{r},{v})") for s in self.NS for i in [s]+self.NC for j in [s]+self.NC for r in Rset for v in self.F if i!=j}
        y = {(v,s): model.addVar(vtype = gp.GRB.BINARY, name = f"y^{s}({v})") for s in self.NS for v in self.F}
        U = {(i,v,s): model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"U^{s}({i},{v})") for s in self.NS for i in [s]+self.NC for v in self.F}
        E = {(i,v,s): model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"E^{s}({i},{v})") for s in self.NS for i in [s]+self.NC for v in self.F}
        T = {i: model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"T({i})") for i in self.NS+self.NC}
        o = {i: model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"T({i})") for i in self.NC}

        #Objective function
        model.setObjective(gp.quicksum(self.h[v]*y0[v] for v in self.T) + gp.quicksum(self.h[v]*y[v,s] for s in self.NS for v in self.F)
                        + gp.quicksum(self.d[i,j]*self.c[v]*x0[i,j,v] for i in Nd_NS for j in Nd_NS for v in self.T if i!=j)
                        + gp.quicksum(self.dr[i,j,r]*self.c[v]*x[i,j,r,v,s] for s in self.NS for i in [s]+self.NC for j in [s]+self.NC for r in Rset for v in self.F if i!=j)
                        + gp.quicksum(self.p[i]*o[i] for i in self.NC), gp.GRB.MINIMIZE)

        #Constraints
        #Assignament
        for i in self.NC:
            model.addConstr(gp.quicksum(w[i,s] for s in self.NS) == 1) #Satellite - Customer

        for s in self.NS:
            model.addConstr(gp.quicksum(self.q[i]*w[i,s] for i in self.NC) <= Mq*a[s]) #Capacity

        #First echelon
        for s in self.NS:
            model.addConstr(gp.quicksum(x0[i,s,v] for i in Nd_NS if i!=s for v in self.T) == a[s]) #Arrives at satellite
            model.addConstr(gp.quicksum(x0[s,i,v] for i in Nd_NS if i!=s for v in self.T) == a[s]) #Departs from satellite
            for v in self.T:
                model.addConstr(gp.quicksum(x0[i,s,v] for i in Nd_NS if i!=s) - gp.quicksum(x0[s,j,v] for j in Nd_NS if j!=s) == 0) #Flow conservation (satellites)

        for v in self.T:
            model.addConstr(gp.quicksum(x0[self.Nd,s,v] for s in self.NS) == y0[v]) #Start at deposit
            model.addConstr(gp.quicksum(x0[s,self.Nd,v] for s in self.NS) == y0[v]) #End at deposit
            for i in Nd_NS:
                model.addConstr(U0[i,v] <= self.Q[v]*y0[v]) #Capacity per route
                for j in self.NS:
                    if i!=j:
                        model.addConstr(U0[j,v] >= U0[i,v] + gp.quicksum(self.q[c]*w[c,j] for c in self.NC) - Mq*(1-x0[i,j,v])) #Transported demand

        #Second-echelon
        for v in self.F:
            model.addConstr(gp.quicksum(y[v,s] for s in self.NS) <= 1)

        for s in self.NS:
            model.addConstr(T[s] >= self.T0) #Minimum hour
            for i in self.NC:
                model.addConstr(gp.quicksum(x[j,i,r,v,s] for j in [s]+self.NC if i!=j for r in Rset for v in self.F) == w[i,s]) #Arrives at customer from the satellite
                model.addConstr(gp.quicksum(x[i,j,r,v,s] for j in [s]+self.NC if i!=j for r in Rset for v in self.F) == w[i,s]) #Departs from customer to the satellite
                for v in self.F:
                    model.addConstr(gp.quicksum(x[k,i,r,v,s] for k in [s]+self.NC if i!=k for r in Rset) - gp.quicksum(x[i,j,r,v,s] for j in [s]+self.NC if i!=j for r in Rset) == 0) #Flow conservation (satellites)   
                    
            for v in self.F:
                model.addConstr(gp.quicksum(x[s,i,r,v,s] for i in self.NC for r in Rset) == y[v,s]) #Start at satellite
                model.addConstr(gp.quicksum(x[i,s,r,v,s] for i in self.NC for r in Rset) == y[v,s]) #End at satellite
                for i in [s]+self.NC:
                    model.addConstr(U[i,v,s] <= self.Q[v]*y[v,s]) #Capacity per route
                    model.addConstr(E[i,v,s] <= self.L[v]*y[v,s]) #Battery per route
                    for j in self.NC:
                        if i!=j:
                            model.addConstr(E[j,v,s] <= E[i,v,s] - self.r[v]*self.d[i,j] + Ml*(1-x[i,j,'0',v,s])) #Current battery                         
                            for r in self.NR:
                                model.addConstr(E[j,v,s] <= self.L[v] - self.r[v]*self.d[r,j] + Ml*(1-x[i,j,r,v,s])) #Current battery
                                model.addConstr(E[i,v,s] >= self.r[v]*self.d[i,r]*x[i,j,r,v,s]) #Enough battery to arrive

                            for r in ['0']+self.NR:
                                model.addConstr(U[j,v,s] >= U[i,v,s] + self.q[j] - Mq*(1-x[i,j,r,v,s])) #Transported demand

                for i in self.NC:
                    model.addConstr(E[i,v,s] >= self.r[v]*self.d[i,s]*x[i,s,'0',v,s]) #Enough battery to return
                    model.addConstr(T[i] >= T[s] + self.t[s,i] - M*(1-x[s,i,'0',v,s])) #Cumulative hours
                    for r in self.NR:
                        model.addConstr(self.L[v] >= self.r[v]*self.d[r,s]*x[i,s,r,v,s]) #Enough battery to return
                        model.addConstr(E[i,v,s] >= self.r[v]*self.d[i,r]*x[i,s,r,v,s]) #Enough battery to return

                    for j in self.NC:
                        if i!=j:
                            model.addConstr(T[j] >= T[i] + self.S[i] + self.t[i,j] - M*(1-x[i,j,'0',v,s])) #Cumulative hours 
                            for r in self.NR:
                                model.addConstr(T[j] >= T[i] + self.S[i] + self.t[i,r] + self.t[r,j] + self.eta*(self.L[v]-(E[i,v,s]-self.r[v]*self.d[i,r])) - M*(1-x[i,j,r,v,s])) #Cumulative hours 

        for i in self.NC:
            model.addConstr(T[i] >= self.e[i])
            model.addConstr(T[i] <= self.l[i]+o[i])

        # Addittional constrains
        for v in self.T:
            model.addConstr(U0[self.Nd,v] == 0)

        for s in self.NS:
            for v in self.F:
                model.addConstr(U[s,v,s] == 0)
                model.addConstr(gp.quicksum(x[s,i,r,v,s] for i in self.NC for r in self.NR) == 0)
                        
        model.update()

        try:            
            model.optimize()
        
            self.w = {(i,s): w[i,s].X for s in self.NS for i in self.NC}
            self.a = {s: a[s].X for s in self.NS}
            self.x0 = {(i,j,v): x0[i,j,v].X for i in Nd_NS for j in Nd_NS for v in self.T if i!=j}
            self.y0 = {v: y0[v].X for v in self.T}
            self.x = {(i,j,r,v,s): x[i,j,r,v,s].X for s in self.NS for i in [s]+self.NC for j in [s]+self.NC for r in Rset for v in self.F if i!=j}
            self.y = {(v,s): y[v,s].X for s in self.NS for v in self.F}

            objf = model.ObjVal
            gap = model.MIPGap
            exe_time = model.Runtime
            
        except:
            self.w = None
            self.a = None
            self.x0 = None
            self.y0 = None
            self.x = None
            self.y = None

            objf = float("inf")
            gap = float("inf")
            exe_time = model.Runtime

        return objf, gap, exe_time

    def graph_route(self, objf, gap):
        if objf == float("inf"):
            return False
        
        def extract_routes_1e():
            routes_1e = []
            for v in self.T:
                if self.y0[v] <= 0.9:
                    continue

                node = self.Nd
                route = [node]
                max_steps = len(self.NS) + 2

                for _ in range(max_steps):
                    next_node = None
                    for i in [self.Nd]+self.NS:
                        if i == node:
                            continue
                        if self.x0[node, i, v] > 0.9:
                            next_node = i
                            break

                    if next_node is None or next_node == self.Nd:
                        route.append(self.Nd)
                        break

                    route.append(next_node)
                    node = next_node

                if len(route) > 2:
                    routes_1e.append((v, route))
            return routes_1e
        
        def extract_routes_2e_and_labels():
            routes_2e = []
            arcs_r = {}

            for s in self.NS:
                if self.a[s] <= 0.9:
                    continue

                for v in self.F:
                    if self.y[v, s] <= 0.9:
                        continue

                    node = s
                    route = [s]
                    max_steps = len(self.NC) + 2

                    for _ in range(max_steps):
                        next_node = None
                        chosen_r = '0'

                        for i in [s] + self.NC:
                            if i == node:
                                continue

                            for r in ['0'] + self.NR:
                                if self.x[node, i, r, v, s] > 0.9:
                                    next_node = i
                                    chosen_r = r
                                    break

                            if next_node is not None:
                                break

                        if next_node is not None and chosen_r != '0':
                            arcs_r[(node, next_node)] = chosen_r

                        if next_node is None or next_node == s:
                            route.append(s)
                            break

                        route.append(next_node)
                        node = next_node

                    if len(route) > 2:
                        routes_2e.append((s, v, route))

            return routes_2e, arcs_r

        routes_1e = extract_routes_1e()
        routes_2e, arcs_r = extract_routes_2e_and_labels()

        G = nx.DiGraph()
        G.add_node(self.Nd, tipo="depot")
        for s in self.NS:
            G.add_node(s, tipo="satellite")
        for i in self.NC:
            G.add_node(i, tipo="customer")

        def build_positions():
            pos = {self.Nd: (0.0, 5.0)}
            n_routes2 = max(1, len(routes_2e))
            for r_idx, (s, v, route) in enumerate(routes_2e):
                customers = route[1:-1]
                n_c = len(customers)
                if n_c == 0:
                    continue

                cx = 5.0 * (r_idx - (n_routes2 - 1)/2.0)
                cy = 0.0
                R = 1.5

                for j, node in enumerate(customers):
                    angle = 2*math.pi*(j + 0.5) / n_c
                    pos[node] = (cx + R*math.cos(angle + math.pi/2),
                                 cy + R*math.sin(angle + math.pi/2))

            # Rango X (si no hay clientes, usa depot)
            xs = [p[0] for p in pos.values()]
            xmin = min(xs) - 2
            xmax = max(xs) + 2

            # Colocar satélites en una línea arriba de clientes
            nS = len(self.NS)
            for idx, s in enumerate(self.NS):
                x_s = xmin + (idx + 1)*((xmax - xmin)/(nS + 1))
                pos[s] = (x_s, 3.0)

            return pos, xmin, xmax

        pos, xmin, xmax = build_positions()

        plt.figure(figsize=(12, 6))

        plt.hlines(y=4.0, xmin=xmin, xmax=xmax, linestyles="dashed", colors="gray", linewidth=1.2)
        plt.hlines(y=2.0, xmin=xmin, xmax=xmax, linestyles="dashed", colors="gray", linewidth=1.2)

        plt.text(xmin + 1, 4.5, "Depósito", ha="center", va="bottom", fontsize=10)
        plt.text(xmin + 1, 3.5, "Satélites", ha="center", va="bottom", fontsize=10)
        plt.text(xmin + 1, 1.5, "Clientes", ha="center", va="bottom", fontsize=10)

        nx.draw_networkx_nodes(G, pos, nodelist=[self.Nd], node_shape="s",
                               node_color="darkturquoise", node_size=900,
                               edgecolors="black")
        nx.draw_networkx_nodes(G, pos, nodelist=self.NS, node_shape="D",
                               node_color="yellowgreen", node_size=900,
                               edgecolors="black")
        nx.draw_networkx_nodes(G, pos, nodelist=self.NC, node_shape="o",
                               node_color="lightblue", node_size=900,
                               edgecolors="black")
        nx.draw_networkx_labels(G, pos, font_size=10, font_weight="bold")

        for v, route in routes_1e:
            arcs = list(zip(route[:-1], route[1:]))
            G.add_edges_from(arcs)
            nx.draw_networkx_edges(
                G, pos, edgelist=arcs,
                arrowstyle="-|>", arrowsize=18, width=2.5,
                edge_color="tab:gray", connectionstyle="arc3,rad=0.12",
                style="dashed"
            )

        colors = plt.cm.Dark2(np.linspace(0, 1, max(1, len(routes_2e))))

        for color, (s, v, route) in zip(colors, routes_2e):
            arcs = list(zip(route[:-1], route[1:]))
            G.add_edges_from(arcs)
            nx.draw_networkx_edges(
                G, pos, edgelist=arcs,
                arrowstyle="-|>", arrowsize=18, width=2.5,
                edge_color=[color], connectionstyle="arc3,rad=0.00"
            )

        if arcs_r:
            nx.draw_networkx_edge_labels(
                G, pos, edge_labels=arcs_r, font_size=8, font_color="black",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8)
            )

        plt.text(0.97, 0.97, f"Costs = {objf:.0f}\nGap = {100*gap:.2f}%",
                 transform=plt.gca().transAxes, ha="right", va="bottom",
                 fontsize=14, bbox=dict(boxstyle="round,pad=0.3",
                                        facecolor="white", edgecolor="black",
                                        alpha=0.8))

        plt.title(f"Instancia: {self.instance_name}", fontsize=14)
        plt.axis("off")
        plt.show()

        return True

    def get_route_2nd(self, s, order, wc, wv):
        Q = sum(self.q[i] for i in order)    
        vt = self.FT
        vt_c = [v for v in vt if self.Qv[v] >= Q]
        if not vt_c:
            return None, None, float("inf"), 0

        dmin_r = {}
        for node in order:
            dmin_r[node] = min(self.d[node,r] for r in self.NR)

        base = [s,*order,s]

        arc_candidates = {}
        for i, j in zip(base, base[1:]):
            arc_candidates[(i,j)] = sorted(self.NR, key=lambda r: self.d[i,r] + self.d[r,j])

        best_route, best_v = None, None
        best_costs, best_duals = float("inf"), 0

        for v in vt_c:
            Lmax = self.Lv[v]
            c = self.cv[v]
            fixed_cost = self.hv[v]
            dual_cost = wv[v]

            route = [s]
            L = Lmax

            feasible = True
            for i, j in zip(base, base[1:]):
                if i != s and j != s:
                    need = self.rv[v]*(self.d[i,j] + dmin_r[j])
                else:
                    need = self.rv[v]*self.d[i,j]

                if L >= need:
                    route.append(j)
                    L -= self.rv[v]*self.d[i,j]
                    continue

                r_best = None
                thr_i = L/self.rv[v]
                thr_j = Lmax/self.rv[v]

                for r in arc_candidates[(i, j)]:
                    if self.d[i,r] < thr_i and self.d[r,j] < thr_j:
                        r_best = r
                        break

                if r_best is None:
                    feasible = False
                    break

                route.append(r_best)
                L = Lmax - self.rv[v]*self.d[r_best,j]
                route.append(j)

            if not feasible:
                continue

            costs = fixed_cost
            duals = dual_cost
            Tcur = self.T0
            L = Lmax

            for i, j in zip(route, route[1:]):
                dij = self.d[i,j]

                Tcur += self.t[i, j]
                L -= self.rv[v]*dij
                costs += c*dij

                if j in self.NC:
                    duals += wc[j]

                    if Tcur < self.e[j]:
                        Tcur = self.e[j]
                    elif Tcur > self.l[j]:
                        costs += self.p[j]*(Tcur - self.l[j])

                    Tcur += self.S[j]
                
                if j in self.NR:
                    Tcur += self.eta*(Lmax - L)
                    L = Lmax

            if (costs - duals) < (best_costs - best_duals):
                best_route, best_v = route, v
                best_costs, best_duals = costs, duals

        if best_route is None:
            return None, None, float("inf"), 0

        return best_route, best_v, best_costs, best_duals
    
    def get_route_1st(self, order, wc, wv, q):
        Q = sum(q[s] for s in order)
        vt = self.TT
        vt_c = [v for v in vt if self.Qv[v] >= Q]
        if not vt_c:
            return None, None, float("inf"), 0

        route = [self.Nd, *order, self.Nd]

        best_route, best_v = None, None
        best_costs, best_duals = float("inf"), 0

        for v in vt_c:
            c = self.cv[v]
            costs = self.hv[v]
            duals = wv[v]

            for i, j in zip(route, route[1:]):
                dij = self.d[i,j]
                costs += c*dij

                if j in self.NC:
                    duals += wc[j]

            if (costs - duals) < (best_costs - best_duals):
                best_route, best_v = route, v
                best_costs, best_duals = costs, duals

        if best_route is None:
            return None, None, float("inf"), 0

        return best_route, best_v, best_costs, best_duals
            
    def LNS_2nd(self, s, w, wc, wv):
        C = [i for i in self.NC if w[i,s] > 0.9]
        if not C:
            return None, None, 0

        C_in, C_out = [], []
        for i in C:
            (C_in if random.random() < 0.5 else C_out).append(i)

        random.shuffle(C_in)
        random.shuffle(C_out)

        best_solution = self.get_route_2nd(s, C_in, wc, wv)

        if best_solution[0] is None:
            max_moves = len(C_in)
            moves = 0
            while moves < max_moves and C_in:
                c = random.choice(C_in)
                C_in.remove(c)
                C_out.append(c)
                best_solution = self.get_route_2nd(s, C_in, wc, wv)
                if best_solution[0] is not None:
                    break
                moves += 1
        
        if best_solution[0] is None:
            return None, None, None

        p = self.p2
        max_not_increase = min(self.max2, round(math.e*math.factorial(len(C))))
        not_increase = 0

        while not_increase < max_not_increase:
            if C_in: #Destroy
                n = max(1, round(random.random()*p*len(C_in)))
                n = min(n, len(C_in))
                rnd = random.random()

                if rnd < 1/3:
                    seed = random.choice(C_in)
                    C_sorted = sorted(C_in, key=lambda c: self.d[seed,c])
                    C_remove = C_sorted[:n]
                elif rnd < 2/3:
                    avg_dist = {}
                    m = len(C_in)
                    for c in C_in:
                        a_dist = 0.0
                        for k in C_in:
                            a_dist += self.d[c,k]
                        avg_dist[c] = a_dist/(m-1) if m > 1 else 0.0

                    C_sorted = sorted(C_in, key=lambda c: avg_dist[c], reverse=True)
                    C_remove = C_sorted[:n]
                else:
                    C_remove = random.sample(C_in, n)

                for c in C_remove: 
                    C_in.remove(c) 
                    C_out.append(c)
                
            # Repair
            random.shuffle(C_out)

            curr_solution = self.get_route_2nd(s, C_in, wc, wv)
            curr_C = C_in.copy()
            C_out_ = C_out.copy()

            for c_ in C_out:
                for i in range(len(C_in) + 1):
                    C_new = C_in[:i] + [c_] + C_in[i:]
                    new_solution = self.get_route_2nd(s, C_new, wc, wv)

                    if new_solution[2] - new_solution[3] < curr_solution[2] - curr_solution[3]:
                        curr_solution, curr_C = new_solution, C_new.copy()

                C_in = curr_C.copy()

            C_out = [c for c in C_out_ if c not in C_in]
   
            if curr_solution[2] - curr_solution[3] < best_solution[2] - best_solution[3]:
                best_solution = curr_solution
                not_increase = 0
            else:
                not_increase += 1

        if best_solution[2] - best_solution[3] < 0:
            return best_solution[0], best_solution[1], best_solution[2]
        else:
            return None, None, None
        
    def LNS_1st(self, w, wc, wv):
        a = {s: 0 for s in self.NS}
        for s in self.NS:
            for i in self.NC:
                if w[i,s] > 0.9:
                    a[s] = 1
        S = [s for s in self.NS if a[s] > 0.9]
        Q = {s: sum(self.q[i]*w[i,s] for i in self.NC) for s in S}

        if not S:
            return None, None, 0

        S_in, S_out = [], []
        for i in S:
            (S_in if random.random() < 0.5 else S_out).append(i)

        random.shuffle(S_in)
        random.shuffle(S_out)

        best_solution = self.get_route_1st(S_in, wc, wv, Q)

        if best_solution[0] is None:
            max_moves = len(S_in)
            moves = 0
            while moves < max_moves and S_in:
                s = random.choice(S_in)
                S_in.remove(s)
                S_out.append(s)
                best_solution = self.get_route_1st(S_in, wc, wv, Q)
                if best_solution[0] is not None:
                    break
                moves += 1
        
        if best_solution[0] is None:
            return None, None, None

        p = self.p1
        max_not_increase = min(self.max1, round(math.e*math.factorial(len(S))))
        not_increase = 0

        while not_increase < max_not_increase:
            if S_in: #Destroy
                n = max(1, round(random.random()*p*len(S_in)))
                n = min(n, len(S_in))

                rnd = random.random()

                if rnd < 1/3:
                    seed = random.choice(S_in)
                    S_sorted = sorted(S_in, key=lambda c: self.d[seed,c])
                    S_remove = S_sorted[:n]
                elif rnd < 2/3:
                    avg_dist = {}
                    m = len(S_in)
                    for c in S_in:
                        a_dist = 0.0
                        for k in S_in:
                            if k != c:
                                a_dist += self.d[c,k]
                        avg_dist[c] = a_dist/(m-1) if m > 1 else 0.0

                    S_sorted = sorted(S_in, key=lambda s: avg_dist[s], reverse=True)
                    S_remove = S_sorted[:n]
                else:
                    S_remove = random.sample(S_in, n)

                for s in S_remove: 
                    S_in.remove(s) 
                    S_out.append(s)
                
            # Repair
            random.shuffle(S_out)

            curr_solution = self.get_route_1st(S_in, wc, wv, Q)
            curr_S = S_in.copy()
            S_out_ = S_out.copy()

            for s_ in S_out:
                for s in range(len(S_in) + 1):
                    S_new = S_in[:s] + [s_] + S_in[s:]
                    new_solution = self.get_route_1st(S_new, wc, wv, Q)

                    if new_solution[2] - new_solution[3] < curr_solution[2] - curr_solution[3]:
                        curr_solution, curr_S = new_solution, S_new.copy()

                S_in = curr_S.copy()

            S_out = [s for s in S_out_ if s not in S_in]
   
            if curr_solution[2] - curr_solution[3] < best_solution[2] - best_solution[3]:
                best_solution = curr_solution
                not_increase = 0
            else:
                not_increase += 1

        if best_solution[2] - best_solution[3] < 0:
            return best_solution[0], best_solution[1], best_solution[2]
        else:
            return None, None, None

    def set_partitioning_2nd(self, w, env_params):
        with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            env = gp.Env(params=env_params)

        #Model
        sp_model = gp.Model("Subproblem: Set partitioning", env=env)
        sp_model.Params.OutputFlag = 0

        #Variables
        x = {(r,s): sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, ub = 1, name = f"x^{s}({r})") for s in self.NS for r in self.R[s]}
        y = {v: sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"y({v})") for v in self.FT}

        #Objective function
        M = 100*len(self.NC)*max(self.C.values())
        sp_model.setObjective(gp.quicksum(self.C[r,s]*x[r,s] for s in self.NS for r in self.R[s]) + gp.quicksum(M*y[v] for v in self.FT), gp.GRB.MINIMIZE)

        #Constraints
        LocCtr_c = {}
        LocCtr_v = {}

        for s in self.NS:
            for i in self.NC:
                LocCtr_c[i,s] = sp_model.addConstr(gp.quicksum(self.A[i,r,s]*x[r,s] for r in self.R[s]) == w[i,s])

        for v in self.FT:
            LocCtr_v[v] = sp_model.addConstr(gp.quicksum(self.b[r,v,s]*x[r,s] for s in self.NS for r in self.R[s]) <= self.Vq[v] + y[v])

        Optimal = False

        while not Optimal:
            sp_model.update()
            sp_model.optimize()

            Optimal = True
            
            for s in self.NS:
                wc = {i: LocCtr_c[i,s].Pi for i in self.NC}
                wv = {v: LocCtr_v[v].Pi for v in self.FT}

                new_r, veh, cost = self.LNS_2nd(s, w, wc, wv)

                if new_r is not None:
                    r_id = f"r{len(self.R[s])}"
                    self.R[s].append(r_id)
                    self.Rd[r_id,s] = new_r
                    self.C[r_id,s] = cost
                    Optimal = False

                    for v in self.FT:
                        self.b[r_id,v,s] = 0 if v != veh else 1

                    for i in self.NC:
                        self.A[i,r_id,s] = 1 if i in new_r else 0

                    newCol = gp.Column()

                    for i in self.NC:
                        newCol.addTerms(self.A[i,r_id,s], LocCtr_c[i,s])

                    for v in self.FT:
                        newCol.addTerms(self.b[r_id,v,s], LocCtr_v[v])

                    x[r_id,s] = sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, ub = 1, obj=cost, name = f"x^{s}({r_id})", column = newCol)

            if Optimal:
                for s in self.NS:
                    for r in self.R[s]:
                        x[r,s].setAttr("vtype", gp.GRB.BINARY)
                
                sp_model.update()
                sp_model.optimize()

                if sum(y[v].X for v in self.FT) > 0 or sp_model.ObjVal > (len(self.NC)*max(self.h.values()))**2:
                    return {s: None for s in self.NS}, float("inf")

                costs = sp_model.ObjVal
                Optimal = True

                r_ = {}
                for s in self.NS:
                    r_s = []
                    for r in self.R[s]:
                        if x[r,s].X > 0.9:
                            r_s.append(r)
                    r_[s] = r_s
                
                return r_, costs
    
    def set_partitioning_1st(self, w, env_params):
        with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            env = gp.Env(params=env_params)

        #Model
        sp_model = gp.Model("Subproblem: Set partitioning", env=env)
        sp_model.Params.OutputFlag = 0

        #Variables
        Q = {s: sum(self.q[i]*w[i,s] for i in self.NC) for s in self.NS}
        x = {}
        R_ = []
        for r in self.R[self.Nd]:
            Rd = self.Rd[r,self.Nd]
            q = sum(Q[s] for s in Rd[1:-1])

            cap_c = max([self.Qv[v] for v in self.TT if self.b[r, v, self.Nd] == 1], default=-1)

            if q <= cap_c:
                R_.append(r)
                x[r] = sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, ub = 1, name = f"x({r})")

        y = {v: sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, name = f"y({v})") for v in self.TT}

        #Objective function
        M = 100*len(self.NS)*max(self.C.values())
        sp_model.setObjective(gp.quicksum(self.C[r,self.Nd]*x[r] for r in R_) + gp.quicksum(M*y[v] for v in self.TT), gp.GRB.MINIMIZE)

        #Constraints
        LocCtr_c = {}
        LocCtr_v = {}

        a = {s: 0 for s in self.NS}
        for s in self.NS:
            for i in self.NC:
                if w[i,s] > 0.9:
                    a[s] = 1

        for s in self.NS:
            LocCtr_c[s] = sp_model.addConstr(gp.quicksum(self.A[s,r,self.Nd]*x[r] for r in R_) == a[s])

        for v in self.TT:
            LocCtr_v[v] = sp_model.addConstr(gp.quicksum(self.b[r,v,self.Nd]*x[r] for r in R_) <= self.Vq[v] + y[v])

        Optimal = False

        while not Optimal:
            sp_model.update()
            sp_model.optimize()

            Optimal = True
            
            wc = {s: LocCtr_c[s].Pi for s in self.NS}
            wv = {v: LocCtr_v[v].Pi for v in self.TT}

            new_r, veh, cost = self.LNS_1st(w, wc, wv)

            if new_r is not None:
                r_id = f"r{len(self.R[self.Nd])}"
                self.R[self.Nd].append(r_id)
                R_.append(r_id)
                self.Rd[r_id,s] = new_r
                self.C[r_id,s] = cost
                Optimal = False

                for v in self.TT:
                    self.b[r_id,v,self.Nd] = 0 if v != veh else 1

                for s in self.NS:
                    self.A[s,r_id,self.Nd] = 1 if s in new_r else 0

                newCol = gp.Column()

                for s in self.NS:
                    newCol.addTerms(self.A[s,r_id,self.Nd], LocCtr_c[s])

                for v in self.TT:
                    newCol.addTerms(self.b[r_id,v,self.Nd], LocCtr_v[v])

                x[r_id] = sp_model.addVar(vtype = gp.GRB.CONTINUOUS, lb = 0, ub = 1, obj=cost, name = f"x^{s}({r_id})", column = newCol)

        for s in self.NS:
            for r in R_:
                x[r].setAttr("vtype", gp.GRB.BINARY)
                
        sp_model.update()
        sp_model.optimize()

        if sum(y[v].X for v in self.TT) > 0:
            return None, float("inf")

        costs = sp_model.ObjVal
        
        r_ = []
        for r in R_:
            if x[r].X > 0.9:
                r_.append(r)
                
        return r_, costs

    def init_math(self, env_params, maxQ):
        with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            env = gp.Env(params=env_params)

        #Model
        w_model = gp.Model("Initialize", env=env)
        w_model.Params.OutputFlag = 0

        #Variables
        w = {(i,s): w_model.addVar(vtype = gp.GRB.BINARY, lb = 0, ub = 1, name = f"w^{i}({s})") for i in self.NC for s in self.NS}

        #Objective function
        w_model.setObjective(gp.quicksum(self.d[s,i]*w[i,s] for i in self.NC for s in self.NS), gp.GRB.MINIMIZE)

        #Constraints
        for i in self.NC:
            w_model.addConstr(gp.quicksum(w[i,s] for s in self.NS) == 1) #Satellite - Customer

        for s in self.NS:
            w_model.addConstr(gp.quicksum(self.q[i]*w[i,s] for i in self.NC) <= maxQ) #Capacity

        w_model.update()
        w_model.optimize()
            
        w_ = {(i,s): round(w[i,s].X) for s in self.NS for i in self.NC}
                
        return w_
    
    def matheuristic(self, params, exe_time):
        start = time.perf_counter()

        r = []
        for s in self.NS:
            for v in self.TT:
                rdx = len(r) if r else 0
                r_id = f"r{rdx}"
                    
                r.append(r_id)
                self.Rd[r_id,self.Nd] = [self.Nd,s,self.Nd]
                self.C[r_id,self.Nd] = self.hv[v] + self.cv[v]*(self.d[self.Nd,s] + self.d[s,self.Nd])

                for v2 in self.TT:
                    self.b[r_id,v2,self.Nd] = 0 if v2 != v else 1

                for i in self.NS:
                    self.A[i,r_id,self.Nd] = 0 if i != s else 1
        self.R[self.Nd] = r

        for s in self.NS:
            r = []
            for i in self.NC:
                for v in self.FT:
                    L_ = (self.d[s,i] + self.d[i,s])*self.rv[v]

                    rdx = len(r) if r else 0
                    r_id = f"r{rdx}"
                    
                    r.append(r_id)
                    self.Rd[r_id,s] = [s,i,s]

                    self.C[r_id,s] = (len(self.NC)*max(self.h.values()))**2 if L_ > self.Lv[v] else self.hv[v] + self.cv[v]*(self.d[s,i] + self.d[i,s])

                    for v2 in self.FT:
                        self.b[r_id,v2,s] = 0 if v2 != v else 1
                
                    for i2 in self.NC:
                        self.A[i2,r_id,s] = 0 if i2 != i else 1
            self.R[s] = r

        # Initialize w's
        q_ = {s: 0 for s in self.NS}
        maxQ = max(self.Q.values())
        w = self.init_math(params, maxQ)

        rs, z = self.set_partitioning_2nd(w, params) 
        r, z_ = self.set_partitioning_1st(w, params)

        rs[self.Nd] = r
        z += z_

        best_z = z
        best_rs = rs.copy()

        improved = True

        while improved:
            now = time.perf_counter()
            improved = False

            if now - start > exe_time:
                end = now
                return best_z, best_rs, end - start
            
            sat = {}
            q = {s: 0 for s in self.NS}

            for i in self.NC:
                for s in self.NS:
                    if w[i,s] > 0.9:
                        sat[i] = s
                        q[s] += self.q[i]

            z_change = float("inf")
            best_change = None

            for i in self.NC:
                curr_s = sat[i]

                for s in self.NS:
                    if time.perf_counter() - start < exe_time:
                        if s != curr_s and q[s] + self.q[i] <= maxQ:
                            z = {}
                            rs = {}

                            w[i,s] = 1
                            w[i,curr_s] = 0

                            if time.perf_counter() - start < exe_time:
                                rs, z = self.set_partitioning_2nd(w, params) 
                            else:
                                rs, z = None, float("inf")

                            if time.perf_counter() - start < exe_time:
                                r, z_ = self.set_partitioning_1st(w, params)  
                                rs[self.Nd] = r
                                z += z_  
                            else:
                                rs, z = None, float("inf")
                                    
                            if z < z_change:
                                z_change = z
                                best_change = w.copy()
                                r_change = rs.copy()

                            w[i,s] = 0
                            w[i,curr_s] = 1

            for i, j in itertools.combinations(self.NC, 2):
                if time.perf_counter() - start < exe_time:
                    si = sat[i]
                    sj = sat[j]

                    if si != sj and q[si] - self.q[i] + self.q[j] <= maxQ and q_[sj] - self.q[j] + self.q[i] <= maxQ:
                        z = {}
                        rs = {}

                        w[i,si], w[i,sj] = 0, 1
                        w[j,si], w[j,sj] = 1, 0

                        if time.perf_counter() - start < exe_time:
                            rs, z = self.set_partitioning_2nd(w, params) 
                        else:
                            rs, z = None, float("inf")

                        if time.perf_counter() - start < exe_time:
                            r, z_ = self.set_partitioning_1st(w, params)  
                            rs[self.Nd] = r
                            z += z_  
                        else:
                            rs, z = None, float("inf")

                        if z < z_change:
                            z_change = z
                            best_change = w.copy()
                            r_change = rs.copy()

                        w[i,si], w[i,sj] = 1, 0
                        w[j,si], w[j,sj] = 0, 1

            if z_change < best_z:
                best_z = z_change
                w = best_change.copy()
                best_rs = r_change.copy()
                improved = True

        end = time.perf_counter()

        return best_z, best_rs, end - start

In [2]:
params = {
    "WLSACCESSID": '84a79f6b-88c8-4dc6-a15d-043da3f6e5f2',
    "WLSSECRET": '59749acc-3ea2-45b9-8322-3f3ecdd7c204',
    "LICENSEID": 939786
}

folder = "H1/Small"
file = "I1-10-1-3-H1"

e2evrp = E2EVRP()
e2evrp.read_instance(folder, file)
of, g, t= e2evrp.milp_model(params, 1800, 1)
e2evrp.graph_route(of, g)

Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 1800
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-10870H CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  1800

Academic license 939786 - for non-commercial use only - registered to ca___@uniandes.edu.co
Optimize a model with 6383 rows, 1899 columns and 23820 nonzeros
Model fingerprint: 0x448f466c
Variable types: 115 continuous, 1784 integer (1784 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+04]
  Objective range  [1e+00, 1e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+04]
Presolve removed 3317 rows and 521 columns
Presolve time: 0.20s
Presolved: 3066 rows, 1378 columns, 26591 nonzeros
Variable types: 100 continuous, 1278 integer (1238 binary)

Root relaxation: objective 1.375000e+03, 44 iterations, 

False

In [28]:
params = {
    "WLSACCESSID": '84a79f6b-88c8-4dc6-a15d-043da3f6e5f2',
    "WLSSECRET": '59749acc-3ea2-45b9-8322-3f3ecdd7c204',
    "LICENSEID": 939786
}

folder = "H1/Small"
file = "I1-10-1-3-H1"

e2evrp = E2EVRP()
e2evrp.read_instance(folder, file)
z, rs, ttime = e2evrp.matheuristic(params, 1800)

if z == float("inf"):
    print("Infeasible")
else:
    print("Cost",z)
    for s in [e2evrp.Nd]+e2evrp.NS:
        for r in rs[s]:
            print(s, r, e2evrp.Rd[r,s])

Infeasible


In [23]:
e2evrp.C

{('r0', 'D0'): 496,
 ('r0', 'S0'): 390,
 ('r1', 'S0'): 408,
 ('r2', 'S0'): 504,
 ('r3', 'S0'): 80,
 ('r4', 'S0'): 362,
 ('r5', 'S0'): 420,
 ('r6', 'S0'): 264,
 ('r7', 'S0'): 2250000,
 ('r8', 'S0'): 656,
 ('r9', 'S0'): 2250000,
 ('r10', 'S0'): 6834.0,
 ('r11', 'S0'): 6260.0,
 ('r12', 'S0'): 1478.0,
 ('r13', 'S0'): 5498.0,
 ('r14', 'S0'): 3021.0,
 ('r15', 'S0'): 2636.0,
 ('r16', 'S0'): 3738.0,
 ('r17', 'S0'): 6231.0,
 ('r18', 'S0'): 7355.0,
 ('r19', 'S0'): 4233.0,
 ('r20', 'S0'): 4548.0,
 ('r21', 'S0'): 4774.0,
 ('r22', 'S0'): 872.0,
 ('r23', 'S0'): 1011.0,
 ('r24', 'S0'): 2956.0,
 ('r25', 'S0'): 3653.0,
 ('r26', 'S0'): 905.0,
 ('r27', 'S0'): 3039.0,
 ('r28', 'S0'): 688.0,
 ('r29', 'S0'): 3963.0,
 ('r30', 'S0'): 3256.0,
 ('r31', 'S0'): 2782.0,
 ('r32', 'S0'): 2286.0,
 ('r33', 'S0'): 4398.0,
 ('r34', 'S0'): 1924.0}

In [ ]:
params = {
    "WLSACCESSID": '84a79f6b-88c8-4dc6-a15d-043da3f6e5f2',
    "WLSSECRET": '59749acc-3ea2-45b9-8322-3f3ecdd7c204',
    "LICENSEID": 939786
}

workbook = load_workbook("./Instances.xlsx")
sheet = workbook['Hoja1']

for i in range(150):
    if sheet.cell(2+i, 10).value is None:
        instance = sheet.cell(2+i, 1).value
        print(instance)
        e2evrp = E2EVRP()
        e2evrp.read_instance(instance)

        if sheet.cell(2+i, 8).value is None:
            objf, gap, exe_time = e2evrp.milp_model(params, 1800)

            sheet.cell(2+i, 6, value=objf)
            sheet.cell(2+i, 7, value=gap)
            sheet.cell(2+i, 8, value=exe_time)
            workbook.save("./Instances.xlsx")
            print("MILP", objf, gap, exe_time)
            
        if sheet.cell(2+i, 10).value is None:
            objf_mh, exe_time_mh = e2evrp.matheuristic(params, 1800)
            sheet.cell(2+i, 9, value=objf_mh)
            sheet.cell(2+i, 10, value=exe_time_mh)
            workbook.save("./Instances.xlsx")

            print("Math", objf_mh, exe_time_mh)